# Phase 10 — Predictive Feature Preparation
## Australian Stock Market Analytics

### Goal
Build a time-aware, leakage-conscious feature table from Phase 8 historical prices and Phase 9 forward-return targets.

This notebook creates lagged returns, rolling volatility, moving averages, momentum, and benchmark features where benchmark data is available. It joins features and labels by ticker/date, validates alignment, defines chronological date splits, and saves a model-ready candidate dataset.

**Important**
- Features at date `t` use only prices dated `t` or earlier.
- Forward-return columns are labels, never predictors.
- A row is not necessarily an independent sample: overlapping forward-return horizons create overlapping outcomes.
- This notebook does not train or evaluate a predictive model.
- The target and feature availability should be reviewed before modeling. A valid historical feature dataset is not a guarantee of predictive power.

## 1. Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
print("Imports successful.")

Imports successful.


## 2. Locate Phase 8 and Phase 9 outputs

In [2]:
PROJECT_ROOT = None
price_file = None
target_file = None
benchmark_file = None

for base in [Path.cwd(), *Path.cwd().parents]:
    p = base / "data" / "processed" / "market_data" / "asx_company_daily_prices.csv"
    t = base / "data" / "processed" / "model_targets" / "asx_forward_return_targets_all_rows.csv"
    b = base / "data" / "processed" / "market_data" / "asx200_benchmark_daily.csv"
    if p.exists() and t.exists():
        PROJECT_ROOT = base
        price_file = p
        target_file = t
        benchmark_file = b if b.exists() else None
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find both Phase 8 company prices and Phase 9 targets. "
        "Expected data/processed/market_data/asx_company_daily_prices.csv and "
        "data/processed/model_targets/asx_forward_return_targets_all_rows.csv."
    )

FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "model_features"
REPORT_DIR = PROJECT_ROOT / "reports" / "predictive_feature_preparation"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

prices = pd.read_csv(price_file)
targets = pd.read_csv(target_file)

print("Price data:", price_file, prices.shape)
print("Target data:", target_file, targets.shape)
print("Benchmark available:", benchmark_file is not None)
display(prices.head())
display(targets.head())

Price data: e:\Projects\Australian Stock Market Data Science Project\data\processed\market_data\asx_company_daily_prices.csv (65771, 8)
Target data: e:\Projects\Australian Stock Market Data Science Project\data\processed\model_targets\asx_forward_return_targets_all_rows.csv (65771, 25)
Benchmark available: True


,date,close,high,low,open,volume,yahoo_symbol,ticker
0,2021-10-19,6.45814,6.46741,6.14311,6.14311,5232326,A2M.AX,A2M
1,2021-10-20,6.84730,6.90289,6.48594,6.53227,15237359,A2M.AX,A2M
2,2021-10-21,6.60639,6.84730,6.56933,6.84730,6015791,A2M.AX,A2M
3,2021-10-22,6.38402,6.59713,6.34695,6.59713,6359080,A2M.AX,A2M
4,2021-10-25,6.30063,6.43961,6.27283,6.39328,4083248,A2M.AX,A2M


,date,close,high,low,open,volume,yahoo_symbol,ticker,adj_close,price_for_returns,future_price_1m,forward_return_1m,future_price_3m,forward_return_3m,future_price_6m,forward_return_6m,benchmark_forward_return_1m,benchmark_forward_return_3m,benchmark_forward_return_6m,relative_forward_return_1m,outperforms_benchmark_1m,relative_forward_return_3m,outperforms_benchmark_3m,relative_forward_return_6m,outperforms_benchmark_6m
0,2021-10-19,6.45814,6.46741,6.14311,6.14311,5232326,A2M.AX,A2M,6.45814,6.45814,5.64277,-0.12626,4.97564,-0.22956,4.34558,-0.32712,-0.00068,-0.00575,0.02955,-0.12558,0.00000,-0.22381,0.00000,-0.35666,0.00000
1,2021-10-20,6.84730,6.90289,6.48594,6.53227,15237359,A2M.AX,A2M,6.84730,6.84730,5.68909,-0.16915,4.94784,-0.27740,4.24366,-0.38024,-0.00465,-0.00962,0.00804,-0.16449,0.00000,-0.26778,0.00000,-0.38828,0.00000
2,2021-10-21,6.60639,6.84730,6.56933,6.84730,6015791,A2M.AX,A2M,6.60639,6.60639,5.75395,-0.12903,4.74400,-0.28191,4.16953,-0.36886,-0.00255,-0.03231,-0.01313,-0.12648,0.00000,-0.24960,0.00000,-0.35573,0.00000
3,2021-10-22,6.38402,6.59713,6.34695,6.59713,6359080,A2M.AX,A2M,6.38402,6.38402,5.75395,-0.09869,4.71620,-0.26125,4.06761,-0.36284,-0.00841,-0.03722,-0.02081,-0.09028,0.00000,-0.22403,0.00000,-0.34204,0.00000
4,2021-10-25,6.30063,6.43961,6.27283,6.39328,4083248,A2M.AX,A2M,6.30063,6.30063,5.84661,-0.07206,5.04977,-0.19853,4.04908,-0.35735,-0.00409,-0.06443,-0.01130,-0.06797,0.00000,-0.13410,0.00000,-0.34605,0.00000


## 3. Normalize schemas and validate keys

In [3]:
def normalize_columns(frame):
    frame = frame.copy()
    frame.columns = [str(c).strip().lower() for c in frame.columns]
    return frame

prices = normalize_columns(prices)
targets = normalize_columns(targets)

for frame_name, frame in [("prices", prices), ("targets", targets)]:
    required = {"ticker", "date"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{frame_name} missing required columns: {sorted(missing)}")
    frame["ticker"] = frame["ticker"].astype(str).str.strip().str.upper()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce", utc=True).dt.tz_localize(None)

prices = prices.dropna(subset=["ticker", "date"]).sort_values(["ticker", "date"])
targets = targets.dropna(subset=["ticker", "date"]).sort_values(["ticker", "date"])

if prices.duplicated(["ticker", "date"]).any():
    raise ValueError("Duplicate ticker/date rows in Phase 8 prices; resolve before feature preparation.")
if targets.duplicated(["ticker", "date"]).any():
    raise ValueError("Duplicate ticker/date rows in Phase 9 targets; resolve before feature preparation.")

if "close" not in prices.columns:
    raise ValueError("Phase 8 prices must contain a 'close' column.")
prices["close"] = pd.to_numeric(prices["close"], errors="coerce")
if "adj_close" in prices.columns:
    prices["adj_close"] = pd.to_numeric(prices["adj_close"], errors="coerce")
else:
    prices["adj_close"] = prices["close"]

prices["price_for_features"] = prices["adj_close"].where(
    prices["adj_close"].notna(), prices["close"]
)
prices = prices.loc[
    prices["price_for_features"].notna() & (prices["price_for_features"] > 0)
].copy()

print("Price rows:", len(prices), "| Target rows:", len(targets))
print("Price tickers:", prices["ticker"].nunique(), "| Target tickers:", targets["ticker"].nunique())

Price rows: 65771 | Target rows: 65771
Price tickers: 52 | Target tickers: 52


## 4. Build lagged return and rolling features
All rolling features are calculated independently for each ticker. `shift(1)` ensures the rolling volatility and moving averages use information through the prior completed observation, reducing same-day execution assumptions. Current-day price is retained as a feature; any later modeling should define the prediction time consistently.

In [4]:
prices = prices.sort_values(["ticker", "date"]).reset_index(drop=True)

# One-observation return, then lagged returns for prior sessions.
prices["return_1d"] = prices.groupby("ticker")["price_for_features"].pct_change()
for lag in [1, 5, 21, 63, 126]:
    prices[f"lagged_return_{lag}d"] = prices.groupby("ticker")["return_1d"].shift(lag)

# Rolling volatility from daily returns, shifted by one observation.
for window in [5, 21, 63]:
    prices[f"volatility_{window}d"] = (
        prices.groupby("ticker")["return_1d"]
        .transform(lambda s: s.rolling(window, min_periods=window).std().shift(1))
    )

# Moving averages use prices available up to the previous completed observation.
for window in [5, 21, 50, 100, 200]:
    prices[f"moving_average_{window}d"] = (
        prices.groupby("ticker")["price_for_features"]
        .transform(lambda s: s.rolling(window, min_periods=window).mean().shift(1))
    )
    prices[f"price_to_ma_{window}d"] = (
        prices["price_for_features"] / prices[f"moving_average_{window}d"] - 1
    )

# Momentum features based only on earlier observed prices.
for window in [5, 21, 63, 126]:
    prices[f"momentum_{window}d"] = (
        prices.groupby("ticker")["price_for_features"].pct_change(window).groupby(prices["ticker"]).shift(1)
    )

feature_columns = [
    c for c in prices.columns
    if c.startswith(("lagged_return_", "volatility_", "moving_average_", "price_to_ma_", "momentum_"))
]
display(prices[["ticker", "date", "price_for_features"] + feature_columns].tail(10))

,ticker,date,price_for_features,lagged_return_1d,lagged_return_5d,lagged_return_21d,lagged_return_63d,lagged_return_126d,volatility_5d,volatility_21d,volatility_63d,moving_average_5d,price_to_ma_5d,moving_average_21d,price_to_ma_21d,moving_average_50d,price_to_ma_50d,moving_average_100d,price_to_ma_100d,moving_average_200d,price_to_ma_200d,momentum_5d,momentum_21d,momentum_63d,momentum_126d
65761,WPR,2026-10-06,2.05000,-0.03810,-0.00315,-0.00855,-0.00412,0.01261,0.01970,0.01227,0.01020,2.11600,-0.03119,2.17556,-0.05772,2.30213,-0.10952,2.33076,-0.12046,2.34327,-0.12515,-0.06343,-0.11946,-0.15207,-0.11929
65762,WPR,2026-10-07,2.08000,0.01485,0.01395,-0.00431,0.00413,0.00000,0.02325,0.01306,0.01043,2.09600,-0.00763,2.16488,-0.03921,2.29431,-0.09341,2.32804,-0.10654,2.34113,-0.11154,-0.04651,-0.09868,-0.13593,-0.11733
65763,WPR,2026-10-08,2.08000,0.01463,-0.02294,-0.00866,0.00000,0.00830,0.02342,0.01374,0.01062,2.07600,0.00193,2.15608,-0.03529,2.28649,-0.09031,2.32523,-0.10546,2.33910,-0.11077,-0.04587,-0.08153,-0.12689,-0.10442
65764,WPR,2026-10-09,2.12000,0.00000,-0.01408,-0.01310,-0.00412,0.00000,0.02226,0.01372,0.01062,2.06600,0.02614,2.14822,-0.01314,2.27947,-0.06996,2.32290,-0.08735,2.33716,-0.09292,-0.02347,-0.07351,-0.12689,-0.11179
65765,WPR,2026-10-12,2.09000,0.01923,-0.03810,-0.01327,-0.00413,0.00823,0.02363,0.01439,0.01095,2.07000,0.00966,2.14367,-0.02504,2.27305,-0.08053,2.32078,-0.09944,2.33556,-0.10514,0.00952,-0.04316,-0.10642,-0.09471
65766,WPR,2026-10-13,2.09000,-0.01415,0.01485,0.01794,0.00830,0.00000,0.01384,0.01443,0.01106,2.08400,0.00288,2.13909,-0.02295,2.26622,-0.07776,2.31845,-0.09854,2.33391,-0.10451,0.03465,-0.04401,-0.11541,-0.11480
65767,WPR,2026-10-14,2.06000,0.00000,0.01463,-0.01322,0.00823,0.00816,0.01329,0.01370,0.01098,2.09200,-0.01530,2.13264,-0.03406,2.25881,-0.08801,2.31622,-0.11062,2.33210,-0.11668,0.01951,-0.06085,-0.12269,-0.11480
65768,WPR,2026-10-15,2.09000,-0.01435,0.00000,0.00000,0.00408,0.00405,0.01377,0.01374,0.01101,2.08800,0.00096,2.12616,-0.01701,2.25118,-0.07160,2.31379,-0.09672,2.33033,-0.10313,-0.00962,-0.06194,-0.14234,-0.13457
65769,WPR,2026-10-16,2.08000,0.01456,0.01923,-0.00446,0.00000,0.00000,0.01567,0.01426,0.01119,2.09000,-0.00478,2.12112,-0.01938,2.24357,-0.07291,2.31156,-0.10018,2.32876,-0.10682,0.00481,-0.04828,-0.13339,-0.12551
65770,WPR,2026-10-19,2.08000,-0.00478,-0.01415,0.00000,-0.00406,0.00000,0.01195,0.01426,0.01119,2.08200,-0.00096,2.11606,-0.01704,2.23596,-0.06975,2.30943,-0.09934,2.32728,-0.10625,-0.01887,-0.04858,-0.13754,-0.12970


## 5. Add benchmark features when available

In [5]:
benchmark_features = []
if benchmark_file is not None:
    benchmark = pd.read_csv(benchmark_file)
    benchmark = normalize_columns(benchmark)
    if "date" not in benchmark.columns:
        print("Benchmark file has no date column; skipping benchmark features.")
    else:
        benchmark["date"] = pd.to_datetime(benchmark["date"], errors="coerce", utc=True).dt.tz_localize(None)
        if "adj_close" in benchmark.columns:
            benchmark["adj_close"] = pd.to_numeric(benchmark["adj_close"], errors="coerce")
        if "close" in benchmark.columns:
            benchmark["close"] = pd.to_numeric(benchmark["close"], errors="coerce")
        benchmark["benchmark_price"] = benchmark.get("adj_close", pd.Series(index=benchmark.index, dtype=float))
        if "close" in benchmark.columns:
            benchmark["benchmark_price"] = benchmark["benchmark_price"].where(
                benchmark["benchmark_price"].notna(), benchmark["close"]
            )
        benchmark = benchmark.dropna(subset=["date", "benchmark_price"]).sort_values("date")
        benchmark = benchmark.drop_duplicates("date", keep="last").reset_index(drop=True)
        benchmark["benchmark_return_1d"] = benchmark["benchmark_price"].pct_change()
        for window in [5, 21, 63]:
            benchmark[f"benchmark_return_{window}d_lagged"] = benchmark["benchmark_price"].pct_change(window).shift(1)
            benchmark[f"benchmark_volatility_{window}d"] = (
                benchmark["benchmark_return_1d"].rolling(window, min_periods=window).std().shift(1)
            )
        benchmark_features = [
            c for c in benchmark.columns
            if c.startswith("benchmark_return_") or c.startswith("benchmark_volatility_")
        ]
        benchmark_join = benchmark[["date"] + benchmark_features].copy()
        prices = prices.merge(benchmark_join, on="date", how="left", validate="many_to_one")
        feature_columns += benchmark_features
        print("Benchmark features added:", benchmark_features)
else:
    print("Benchmark file not found; proceeding with company-only features.")

display(prices[["date"] + benchmark_features].head() if benchmark_features else prices[["date"]].head())

Benchmark features added: ['benchmark_return_1d', 'benchmark_return_5d_lagged', 'benchmark_volatility_5d', 'benchmark_return_21d_lagged', 'benchmark_volatility_21d', 'benchmark_return_63d_lagged', 'benchmark_volatility_63d']


,date,benchmark_return_1d,benchmark_return_5d_lagged,benchmark_volatility_5d,benchmark_return_21d_lagged,benchmark_volatility_21d,benchmark_return_63d_lagged,benchmark_volatility_63d
0,2021-10-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-10-20,0.00526,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-10-21,0.00023,NaN,NaN,NaN,NaN,NaN,NaN
3,2021-10-22,0.00001,NaN,NaN,NaN,NaN,NaN,NaN
4,2021-10-25,0.00344,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Merge features with Phase 9 labels

In [6]:
# Keep labels from Phase 9; do not calculate them again here.
label_columns = [
    c for c in targets.columns
    if c.startswith((
        "future_price_", "forward_return_", "benchmark_future_price_",
        "benchmark_forward_return_", "relative_forward_return_",
        "outperforms_benchmark_"
    ))
]
target_keep = targets[["ticker", "date"] + label_columns].copy()

model_table = prices.merge(
    target_keep,
    on=["ticker", "date"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_target")
)

print("Merged model table:", model_table.shape)
print("Target columns:", label_columns)
display(model_table[["ticker", "date"] + feature_columns[:5] + label_columns[:3]].head())

Merged model table: (65771, 55)
Target columns: ['future_price_1m', 'forward_return_1m', 'future_price_3m', 'forward_return_3m', 'future_price_6m', 'forward_return_6m', 'benchmark_forward_return_1m', 'benchmark_forward_return_3m', 'benchmark_forward_return_6m', 'relative_forward_return_1m', 'outperforms_benchmark_1m', 'relative_forward_return_3m', 'outperforms_benchmark_3m', 'relative_forward_return_6m', 'outperforms_benchmark_6m']


,ticker,date,lagged_return_1d,lagged_return_5d,lagged_return_21d,lagged_return_63d,lagged_return_126d,future_price_1m,forward_return_1m,future_price_3m
0,A2M,2021-10-19,NaN,NaN,NaN,NaN,NaN,5.64277,-0.12626,4.97564
1,A2M,2021-10-20,NaN,NaN,NaN,NaN,NaN,5.68909,-0.16915,4.94784
2,A2M,2021-10-21,0.06026,NaN,NaN,NaN,NaN,5.75395,-0.12903,4.74400
3,A2M,2021-10-22,-0.03518,NaN,NaN,NaN,NaN,5.75395,-0.09869,4.71620
4,A2M,2021-10-25,-0.03366,NaN,NaN,NaN,NaN,5.84661,-0.07206,5.04977


## 7. Define predictor and target columns explicitly

In [7]:
# Predictors are historical features only. Exclude identifiers, raw future prices,
# forward-return labels, and any column whose name indicates future outcomes.
identifier_columns = ["ticker", "date"]
target_prefixes = (
    "future_price_", "forward_return_", "benchmark_future_price_",
    "benchmark_forward_return_", "relative_forward_return_", "outperforms_benchmark_"
)
predictor_columns = [
    c for c in feature_columns
    if c in model_table.columns and not c.startswith(target_prefixes)
]

target_name = "forward_return_3m"
if target_name not in model_table.columns:
    raise ValueError(f"Required target {target_name} not found. Re-run Phase 9.")

leakage_check = [
    c for c in predictor_columns
    if any(token in c.lower() for token in ["future", "forward_return", "outperforms_benchmark"])
]
if leakage_check:
    raise ValueError(f"Potential target leakage in predictors: {leakage_check}")

print("Predictor columns:", predictor_columns)
print("Primary target:", target_name)
print("Number of predictors:", len(predictor_columns))

Predictor columns: ['lagged_return_1d', 'lagged_return_5d', 'lagged_return_21d', 'lagged_return_63d', 'lagged_return_126d', 'volatility_5d', 'volatility_21d', 'volatility_63d', 'moving_average_5d', 'price_to_ma_5d', 'moving_average_21d', 'price_to_ma_21d', 'moving_average_50d', 'price_to_ma_50d', 'moving_average_100d', 'price_to_ma_100d', 'moving_average_200d', 'price_to_ma_200d', 'momentum_5d', 'momentum_21d', 'momentum_63d', 'momentum_126d', 'benchmark_return_1d', 'benchmark_return_5d_lagged', 'benchmark_volatility_5d', 'benchmark_return_21d_lagged', 'benchmark_volatility_21d', 'benchmark_return_63d_lagged', 'benchmark_volatility_63d']
Primary target: forward_return_3m
Number of predictors: 29


## 8. Check feature availability and missingness

In [8]:
feature_missingness = pd.DataFrame({
    "feature": predictor_columns,
    "missing_count": [int(model_table[c].isna().sum()) for c in predictor_columns],
    "missing_pct": [float(model_table[c].isna().mean() * 100) for c in predictor_columns]
}).sort_values("missing_pct", ascending=False)
display(feature_missingness)
feature_missingness.to_csv(REPORT_DIR / "feature_missingness.csv", index=False)

label_summary = pd.DataFrame([{
    "total_rows": len(model_table),
    "rows_with_3m_target": int(model_table[target_name].notna().sum()),
    "rows_without_3m_target": int(model_table[target_name].isna().sum()),
    "unique_tickers": int(model_table["ticker"].nunique()),
    "date_min": str(model_table["date"].min()),
    "date_max": str(model_table["date"].max())
}])
display(label_summary)
label_summary.to_csv(REPORT_DIR / "label_coverage_summary.csv", index=False)

,feature,missing_count,missing_pct
16,moving_average_200d,10400,15.81244
17,price_to_ma_200d,10400,15.81244
21,momentum_126d,6604,10.04090
4,lagged_return_126d,6604,10.04090
15,price_to_ma_100d,5200,7.90622
14,moving_average_100d,5200,7.90622
3,lagged_return_63d,3328,5.05998
20,momentum_63d,3328,5.05998
7,volatility_63d,3328,5.05998
27,benchmark_return_63d_lagged,3319,5.04630


,total_rows,rows_with_3m_target,rows_without_3m_target,unique_tickers,date_min,date_max
0,65771,62495,3276,52,2021-10-19 00:00:00,2026-10-19 00:00:00


## 9. Create chronological train/validation/test date windows
Use unique dates, not random row splitting. The default is an ordered 70%/15%/15% split by dates. For later modeling, account for overlapping target horizons: purge or embargo training rows whose forward-label window crosses into validation/test periods.

In [9]:
unique_dates = np.array(sorted(model_table["date"].dropna().unique()))
if len(unique_dates) < 10:
    raise ValueError("Too few unique dates for a meaningful chronological split.")

n_dates = len(unique_dates)
train_cut = int(n_dates * 0.70)
val_cut = int(n_dates * 0.85)
train_end = pd.Timestamp(unique_dates[max(train_cut - 1, 0)])
val_end = pd.Timestamp(unique_dates[max(val_cut - 1, 0)])

def assign_split(date):
    if date <= train_end:
        return "train"
    elif date <= val_end:
        return "validation"
    return "test"

model_table["time_split"] = model_table["date"].apply(assign_split)

split_summary = (
    model_table.groupby("time_split")
    .agg(
        rows=("ticker", "size"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        tickers=("ticker", "nunique"),
        labeled_3m_rows=(target_name, "count")
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
display(split_summary)
split_summary.to_csv(REPORT_DIR / "chronological_split_summary.csv", index=False)

print("Train ends:", train_end)
print("Validation ends:", val_end)
print("Test period starts after:", val_end)

,time_split,rows,first_date,last_date,tickers,labeled_3m_rows
0,train,46011,2021-10-19,2025-04-22,52,46011
1,validation,9880,2025-04-23,2026-01-20,52,9880
2,test,9880,2026-01-21,2026-10-19,52,6604


Train ends: 2025-04-22 00:00:00
Validation ends: 2026-01-20 00:00:00
Test period starts after: 2026-01-20 00:00:00


## 10. Build a labeled candidate table without imputing predictors
For now, retain missing feature values and record them. Imputation/scaling should be fitted on the training period only inside a modeling pipeline to avoid leakage.

In [10]:
labeled = model_table.loc[model_table[target_name].notna()].copy()
labeled = labeled.sort_values(["date", "ticker"]).reset_index(drop=True)

candidate_columns = ["ticker", "date", "time_split"] + predictor_columns + [
    c for c in label_columns if c in labeled.columns
]
candidate_table = labeled[candidate_columns].copy()

print("Labeled rows:", len(candidate_table))
print("Rows by split:")
display(candidate_table["time_split"].value_counts().rename("rows").to_frame())
display(candidate_table.head())

candidate_file = FEATURE_DIR / "asx_3m_return_features_and_targets.csv"
candidate_table.to_csv(candidate_file, index=False)
print("Saved labeled candidate dataset:", candidate_file)

Labeled rows: 62495
Rows by split:


,rows
time_split,
train,46011
validation,9880
test,6604


,ticker,date,time_split,lagged_return_1d,lagged_return_5d,lagged_return_21d,lagged_return_63d,lagged_return_126d,volatility_5d,volatility_21d,volatility_63d,moving_average_5d,price_to_ma_5d,moving_average_21d,price_to_ma_21d,moving_average_50d,price_to_ma_50d,moving_average_100d,price_to_ma_100d,moving_average_200d,price_to_ma_200d,momentum_5d,momentum_21d,momentum_63d,momentum_126d,benchmark_return_1d,benchmark_return_5d_lagged,benchmark_volatility_5d,benchmark_return_21d_lagged,benchmark_volatility_21d,benchmark_return_63d_lagged,benchmark_volatility_63d,future_price_1m,forward_return_1m,future_price_3m,forward_return_3m,future_price_6m,forward_return_6m,benchmark_forward_return_1m,benchmark_forward_return_3m,benchmark_forward_return_6m,relative_forward_return_1m,outperforms_benchmark_1m,relative_forward_return_3m,outperforms_benchmark_3m,relative_forward_return_6m,outperforms_benchmark_6m
0,A2M,2021-10-19,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.64277,-0.12626,4.97564,-0.22956,4.34558,-0.32712,-0.00068,-0.00575,0.02955,-0.12558,0.00000,-0.22381,0.00000,-0.35666,0.00000
1,ACL,2021-10-19,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.28089,0.01152,4.14782,0.27880,4.04745,0.24786,-0.00068,-0.00575,0.02955,0.01220,1.00000,0.28455,1.00000,0.21831,1.00000
2,ADH,2021-10-19,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.69951,-0.07908,2.91637,-0.00510,2.22134,-0.24220,-0.00068,-0.00575,0.02955,-0.07840,0.00000,0.00065,1.00000,-0.27175,0.00000
3,AFG,2021-10-19,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.76002,-0.12132,1.87048,-0.06618,1.59217,-0.20512,-0.00068,-0.00575,0.02955,-0.12065,0.00000,-0.06043,0.00000,-0.23467,0.00000
4,ALD,2021-10-19,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.15545,-0.01525,23.75818,0.01038,25.51123,0.08494,-0.00068,-0.00575,0.02955,-0.01457,0.00000,0.01613,1.00000,0.05539,1.00000


Saved labeled candidate dataset: e:\Projects\Australian Stock Market Data Science Project\data\processed\model_features\asx_3m_return_features_and_targets.csv


## 11. Save feature dictionary and leakage register

In [11]:
feature_dictionary = pd.DataFrame([
    {
        "feature": col,
        "role": "predictor",
        "definition": (
            "Lagged historical return / rolling volatility / moving-average relationship / momentum / benchmark feature"
        ),
        "availability_rule": "Calculated from observations at or before prediction date; rolling summaries shifted by one observation where specified"
    }
    for col in predictor_columns
] + [
    {
        "feature": col,
        "role": "target_or_target_construction_only",
        "definition": "Future outcome or forward-return label created in Phase 9",
        "availability_rule": "Not available at prediction time; never use as predictor"
    }
    for col in label_columns
])
display(feature_dictionary)
feature_dictionary.to_csv(REPORT_DIR / "feature_dictionary_and_leakage_register.csv", index=False)

,feature,role,definition,availability_rule
0,lagged_return_1d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
1,lagged_return_5d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
2,lagged_return_21d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
3,lagged_return_63d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
4,lagged_return_126d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
5,volatility_5d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
6,volatility_21d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
7,volatility_63d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
8,moving_average_5d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...
9,price_to_ma_5d,predictor,Lagged historical return / rolling volatility ...,Calculated from observations at or before pred...


## 12. Final validation and summary

In [12]:
summary = pd.DataFrame([{
    "price_rows": int(len(prices)),
    "model_table_rows": int(len(model_table)),
    "unique_tickers": int(model_table["ticker"].nunique()),
    "predictor_count": len(predictor_columns),
    "primary_target": target_name,
    "labeled_rows": int(model_table[target_name].notna().sum()),
    "unlabeled_rows": int(model_table[target_name].isna().sum()),
    "train_rows": int((model_table["time_split"] == "train").sum()),
    "validation_rows": int((model_table["time_split"] == "validation").sum()),
    "test_rows": int((model_table["time_split"] == "test").sum()),
    "duplicate_ticker_date_rows": int(model_table.duplicated(["ticker", "date"]).sum())
}])
display(summary)
summary.to_csv(REPORT_DIR / "phase10_feature_preparation_summary.csv", index=False)

print("Model feature dataset saved to:", candidate_file)
print("Reports saved to:", REPORT_DIR)
print("PHASE 10 FEATURE PREPARATION COMPLETED.")
print("Before model training, review label coverage, missingness, and time split boundaries.")

,price_rows,model_table_rows,unique_tickers,predictor_count,primary_target,labeled_rows,unlabeled_rows,train_rows,validation_rows,test_rows,duplicate_ticker_date_rows
0,65771,65771,52,29,forward_return_3m,62495,3276,46011,9880,9880,0


Model feature dataset saved to: e:\Projects\Australian Stock Market Data Science Project\data\processed\model_features\asx_3m_return_features_and_targets.csv
Reports saved to: e:\Projects\Australian Stock Market Data Science Project\reports\predictive_feature_preparation
PHASE 10 FEATURE PREPARATION COMPLETED.
Before model training, review label coverage, missingness, and time split boundaries.
